# Fraud model — prior digital-event coverage audit

**Access status: blocked.** On 2026-10-02, Databricks returned `INSUFFICIENT_PERMISSIONS` because the active principal lacks `SELECT` on `workspace.bank_silver.digital_events`. Request that table grant, then rerun this notebook.

This read-only notebook measures whether the table has enough strictly prior customer activity to justify a V3 model iteration. It computes only bounded aggregates on train/validation transactions and never exposes customer, event, transaction, IP, or session identifiers. The final-test period is excluded.

In [ ]:
import json
from pyspark.sql import functions as F, Window

SOURCE_TX = "workspace.bank_silver.transactions"
SOURCE_EVENTS = "workspace.bank_silver.digital_events"
DELTA_VERSION = 1
TRAIN_END = "2025-07-01"
VALIDATION_END = "2026-01-01"

spark.conf.set("spark.sql.session.timeZone", "UTC")
# Validate exact window semantics: lower bound included; same-time and future excluded.
from datetime import datetime, timedelta
T = datetime(2025, 1, 10, 12, 0, 0)
fixture = spark.createDataFrame([
    ("c1", T - timedelta(days=7), 1, 0), ("c1", T - timedelta(days=2), 1, 0),
    ("c1", T - timedelta(hours=12), 1, 0), ("c1", T, 1, 0),
    ("c1", T, 0, 1), ("c1", T + timedelta(seconds=1), 1, 0),
], "history_customer string, event_time timestamp, event_marker long, is_tx int")
fixture = fixture.withColumn("event_time_us", F.expr("unix_micros(event_time)"))
fixture_window = Window.partitionBy("history_customer").orderBy("event_time_us")
fixture_row = (fixture
    .withColumn("prior_24h", F.sum("event_marker").over(fixture_window.rangeBetween(-86_400_000_000, -1)))
    .withColumn("prior_7d", F.sum("event_marker").over(fixture_window.rangeBetween(-604_800_000_000, -1)))
    .where(F.col("is_tx") == 1).first())
assert fixture_row["prior_24h"] == 1 and fixture_row["prior_7d"] == 3, fixture_row
transactions = (
    spark.read.option("versionAsOf", DELTA_VERSION).table(SOURCE_TX)
    .where(F.col("transaction_date") < F.to_timestamp(F.lit(VALIDATION_END)))
    .where(F.col("transaction_date").isNotNull())
    .select("transaction_id", "customer_id", "transaction_date", F.col("is_fraud").cast("double").alias("label"))
    .where(F.col("label").isin(0.0, 1.0))
    .withColumn("split", F.when(F.col("transaction_date") < F.to_timestamp(F.lit(TRAIN_END)), "train").otherwise("validation"))
    .withColumn("history_customer", F.when(F.col("customer_id").isNotNull(), F.col("customer_id")).otherwise(F.concat(F.lit("__missing_tx__"), F.col("transaction_id"))))
    .withColumn("event_marker", F.lit(0).cast("long"))
    .withColumn("sort_us", F.expr("unix_micros(transaction_date)"))
)
events = (
    spark.read.option("versionAsOf", DELTA_VERSION).table(SOURCE_EVENTS)
    .where((F.col("event_date") < F.to_timestamp(F.lit(VALIDATION_END))) & F.col("event_date").isNotNull())
    .select("event_id", "customer_id", "event_date", "event_type")
    .withColumn("history_customer", F.col("customer_id"))
    .withColumn("event_marker", F.lit(1).cast("long"))
    .withColumn("sort_us", F.expr("unix_micros(event_date)"))
)
# Build one sorted history stream; range frames ending at T-1 microsecond exclude simultaneous/future events.
tx_stream = transactions.select(
    "transaction_id", "history_customer", F.col("sort_us").alias("event_time_us"), "event_marker", "label", "split"
).withColumn("is_tx", F.lit(1))
ev_stream = events.where(F.col("history_customer").isNotNull()).select(
    F.lit(None).cast("string").alias("transaction_id"), "history_customer", F.col("sort_us").alias("event_time_us"), "event_marker", F.lit(None).cast("double").alias("label"), F.lit(None).cast("string").alias("split")
).withColumn("is_tx", F.lit(0))
stream = tx_stream.unionByName(ev_stream)
ordered = Window.partitionBy("history_customer").orderBy("event_time_us")
with_counts = (
    stream
    .withColumn("prior_events_24h", F.sum("event_marker").over(ordered.rangeBetween(-86_400_000_000, -1)))
    .withColumn("prior_events_7d", F.sum("event_marker").over(ordered.rangeBetween(-604_800_000_000, -1)))
    .where(F.col("is_tx") == 1)
    .fillna(0, subset=["prior_events_24h", "prior_events_7d"])
)
summary_rows = with_counts.groupBy("split").agg(
    F.count("transaction_id").alias("transactions"), F.sum("label").alias("fraud"),
    F.sum(F.when(F.col("prior_events_24h") > 0, 1).otherwise(0)).alias("with_prior_event_24h"),
    F.sum(F.when(F.col("prior_events_7d") > 0, 1).otherwise(0)).alias("with_prior_event_7d"),
    F.avg("prior_events_24h").alias("mean_prior_events_24h"), F.avg("prior_events_7d").alias("mean_prior_events_7d")
).orderBy("split").collect()
label_rows = with_counts.groupBy("split", (F.col("prior_events_7d") > 0).alias("has_prior_event_7d")).agg(
    F.count("transaction_id").alias("transactions"), F.sum("label").alias("fraud")
).orderBy("split", "has_prior_event_7d").collect()
quality = events.agg(
    F.count("event_id").alias("events"), F.countDistinct("customer_id").alias("customers_with_events"),
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_events"),
    F.min("event_date").cast("string").alias("min_event_date"), F.max("event_date").cast("string").alias("max_event_date")
).first().asDict()
event_types = [r.asDict() for r in events.groupBy("event_type").count().orderBy(F.desc("count"), "event_type").limit(12).collect()]
result = {
    "status": "SUCCESS", "transaction_table": SOURCE_TX, "digital_event_table": SOURCE_EVENTS,
    "delta_version": DELTA_VERSION, "train_end_exclusive": TRAIN_END, "validation_end_exclusive": VALIDATION_END,
    "final_test_used": False, "history_rule": "same customer, [T-7d,T), events at T excluded; event rows with null customer_id do not match",
    "event_quality": quality, "event_type_counts": event_types,
    "coverage_by_split": [r.asDict() for r in summary_rows],
    "fraud_by_prior_event_7d": [r.asDict() for r in label_rows],
    "source_tables_modified": False, "row_level_records_exported": False
}
print(json.dumps(result, indent=2, default=str, allow_nan=False))
dbutils.notebook.exit(json.dumps(result, default=str, allow_nan=False))

## Interpretation

Compare the number and share of transactions with prior digital events across training and validation, then compare fraud prevalence for covered versus uncovered transactions. A low coverage rate means digital-event features cannot support the bulk transaction-scoring population. Coverage is not proof that missing events mean no activity; the event feed is incomplete by design.